In [0]:
%sql
USE CATALOG workspace;

-- Dim Tables:
CREATE OR REPLACE TABLE data_mart.dim_customer AS
SELECT * FROM silver.crm_cust_base;

CREATE OR REPLACE TABLE data_mart.dim_products AS
SELECT * FROM silver.products;

CREATE OR REPLACE TABLE data_mart.dim_stores AS
SELECT * FROM silver.stores_geo;

In [0]:
%sql
USE CATALOG workspace;

-- -- Fact tables
CREATE OR REPLACE TABLE data_mart.fact_inv_levels AS
SELECT * FROM silver.inv_levels;

CREATE OR REPLACE TABLE data_mart.fact_sales AS
(
  SELECT
    sales.trxn_id,
    sales.cust_ref_id,
    sales.date_ref as sale_date,
    sales.product_id,
    sales.store_id,
    sales.qty_sold,
    products.base_cost,
    products.marked_price,
    sales.unit_price as sold_price,
    (products.marked_price - sales.unit_price) * sales.qty_sold as discounted_amount,
    (sales.unit_price - products.base_cost) * sales.qty_sold as profit_amount,
    (sales.unit_price * sales.qty_sold) as total_amount
  FROM
    silver.sales as sales
      join data_mart.dim_products as products
        on sales.product_id = products.product_id
);

CREATE OR REPLACE TABLE data_mart.fact_returns AS
(
  SELECT
    rtn_trans.rtnid_no,
    rtn_trans.orgnl_trxn_id,
    rtn_trans.rtn_date,
    sales.cust_ref_id,
    sales.product_id,
    sales.store_id,
    sales.qty_sold,
    sales.unit_price as sold_price
  FROM silver.rtn_trans as rtn_trans
  JOIN silver.sales as sales
    ON rtn_trans.orgnl_trxn_id = sales.trxn_id
);